In [12]:
import healpy as hp
%matplotlib inline
import matplotlib.pylab as plt
import numpy as np
from rubin_nights import lfa_data
from rubin_nights import connections
import os
from astropy.time import Time, TimeDelta

from astropy.coordinates import SkyCoord
from astroplan import Observer
import astropy.units as u
import pandas as pd

In [13]:
endpoints = connections.get_clients(tokenfile=os.path.join(os.path.expanduser("~"), ".lsst/usdf_rsp"), site="usdf")

In [14]:
#day_obs = Time(Time.now().mjd - 0.5, format='mjd', scale='tai').iso[0:10]
day_obs = "2025-09-18"

queue = 1


day_obs_time = Time(f"{day_obs}T12:00:00", format='isot', scale='tai')
tnow = Time.now()
observer = Observer.at_site('Rubin')
sunset = Time(observer.sun_set_time(day_obs_time, which='next', horizon=-0*u.deg), format='jd')
sunrise = Time(observer.sun_rise_time(day_obs_time, which='next', horizon=-0*u.deg), format='jd')
print(day_obs, 'sunset', sunset.iso,  'sunrise', sunrise.iso, 'now', Time.now().iso)

topic = "lsst.sal.Scheduler.logevent_target"
#topic = "lsst.sal.Scheduler.logevent_largeFileObjectAvailable"
targets = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
for c in ['ra', 'decl', 'skyAngle']:
    targets[c] = targets[c].astype(float)
if len(targets) == 0:
    display(endpoints['efd'].select_top_n(topic, '*', 2, index=queue))
print(len(targets))

topic = "lsst.sal.Scheduler.logevent_observation"
observations = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
for c in ['ra', 'decl', 'rotSkyPos']:
    observations[c] = observations[c].astype(float)
print(len(observations))

topic = "lsst.sal.Scheduler.logevent_largeFileObjectAvailable"
snapshots = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
print(len(snapshots))

if len(targets) > 0 and len(observations) > 0:
    to = pd.merge_asof(
                targets.sort_values("targetId").reset_index("time"),
                observations.sort_values("targetId").reset_index("time"),
                on="targetId",
                left_by=["ra", "decl", "skyAngle"],
                right_by=["ra", "decl", "rotSkyPos"],
                suffixes=("", "_o"),
                allow_exact_matches=True,
                direction="forward",
            )
    to.sort_values(by="time", inplace=True)
    to = to.astype({"targetId": int, "blockId": int, "skyAngle": float})
    print(len(to))
elif len(targets) > 0:
    to = pd.DataFrame(targets.sort_values("targetId").reset_index("time"))
    to['time_o'] = np.nan
    print("targets only")
else:
    to = None


2025-09-18 sunset 2025-09-18 22:33:24.775 sunrise 2025-09-19 10:39:44.092 now 2026-08-14 23:23:33.167
69
40
17
69


In [15]:
snapshots

,byteSize,checkSum,generator,id,mimeType,private_efdStamp,private_identity,private_kafkaStamp,private_origin,private_rcvStamp,private_revCode,private_seqNum,private_sndStamp,salIndex,url,version
time,,,,,,,,,,,,,,,,
2025-09-19 05:58:09.324751+00:00,0,,Scheduler:1,,,1.758261e+09,Scheduler:1,1.758262e+09,13,0,3ba133e0,20,1.758262e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:23:25.696501+00:00,0,,Scheduler:1,,,1.758263e+09,Scheduler:1,1.758263e+09,13,0,3ba133e0,21,1.758263e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:23:36.184977+00:00,0,,Scheduler:1,,,1.758263e+09,Scheduler:1,1.758263e+09,13,0,3ba133e0,22,1.758263e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:25:11.625150+00:00,0,,Scheduler:1,,,1.758263e+09,Scheduler:1,1.758263e+09,13,0,3ba133e0,23,1.758263e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:25:21.118947+00:00,0,,Scheduler:1,,,1.758263e+09,Scheduler:1,1.758263e+09,13,0,3ba133e0,24,1.758263e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:40:40.123650+00:00,0,,Scheduler:1,,,1.758264e+09,Scheduler:1,1.758264e+09,13,0,3ba133e0,25,1.758264e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:40:48.432176+00:00,0,,Scheduler:1,,,1.758264e+09,Scheduler:1,1.758264e+09,13,0,3ba133e0,26,1.758264e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:45:03.587908+00:00,0,,Scheduler:1,,,1.758264e+09,Scheduler:1,1.758264e+09,13,0,3ba133e0,27,1.758264e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0
2025-09-19 06:45:12.815488+00:00,0,,Scheduler:1,,,1.758264e+09,Scheduler:1,1.758264e+09,13,0,3ba133e0,28,1.758264e+09,1,https://s3.cp.lsst.org/rubinobs-lfa-cp/Schedul...,0


In [19]:
snapshots["url"].iloc[0]

'https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2025/09/18/Scheduler:1_Scheduler:1_2025-09-19T05:58:43.595.p'

In [17]:
uri = "https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/07/13/Scheduler:1_Scheduler:1_2026-07-14T10:07:20.305.p"
sched, conditions= lfa_data.get_scheduler_snapshot(uri, at_usdf=False)

ConnectionError: HTTPSConnectionPool(host='s3.cp.lsst.org', port=443): Max retries exceeded with url: /rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/07/13/Scheduler:1_Scheduler:1_2026-07-14T10:07:20.305.p (Caused by NewConnectionError("HTTPSConnection(host='s3.cp.lsst.org', port=443): Failed to establish a new connection: [Errno 61] Connection refused"))